In [1]:
from dotenv import load_dotenv
import os

load_dotenv()


True

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()


# Document Loader

In [129]:
file_path = r"data\rag_chunking.pdf"

In [115]:
from pathlib import Path
from typing import List, Dict, Any, Tuple
import pymupdf
from docx import Document as DocxDocument
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from datetime import datetime
import hashlib
import re
import unicodedata

class OptimizedPreprocessedLoader:
    """RAG loader with page numbers and optimized chunk sizes"""

    def __init__(self, chunk_size=1000, chunk_overlap=200,
                 min_chunk_size=None, max_chunk_size=None,
                 preprocessing_config=None):

        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

        # Set min/max chunk sizes (default: 80-120% of target)
        self.min_chunk_size = min_chunk_size or int(chunk_size * 0.8)
        self.max_chunk_size = max_chunk_size or int(chunk_size * 1.2)

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
        )

        # Default preprocessing config
        self.config = {
            'remove_extra_whitespace': True,
            'remove_extra_newlines': True,
            'normalize_unicode': True,
            'fix_encoding_errors': True,
            'fix_common_ocr_errors': True,
            'remove_headers_footers': True,
            'remove_page_numbers': True,
            'remove_short_lines': True,
        }

        if preprocessing_config:
            self.config.update(preprocessing_config)

    def load_and_split(self, source: str, custom_metadata: Dict = None) -> List[Document]:
        """Load with page tracking and optimized chunking"""

        if source.startswith(('http://', 'https://')):
            return self._load_from_url(source, custom_metadata or {})
        else:
            return self._load_from_file(source, custom_metadata or {})

    def _load_from_url(self, url: str, custom_metadata: Dict) -> List[Document]:
        """Load URL (no page numbers for web content)"""
        try:
            loader = WebBaseLoader(
                web_paths=[url]
            )
            docs = loader.load()

            # Preprocess text
            for doc in docs:
                doc.page_content = self.preprocess_text(doc.page_content)
                doc.metadata.update({
                    'source': url,
                    'source_type': 'url',
                    'file_type': 'html',
                    'loaded_at': datetime.now().isoformat(),
                    'doc_id': self._generate_doc_id(url),
                    **custom_metadata
                })

            # Optimized chunking
            chunks = self._optimized_split(docs)

            return chunks

        except Exception as e:
            raise Exception(f"Failed to load URL: {e}")

    def _load_from_file(self, file_path: str, custom_metadata: Dict) -> List[Document]:
        """Load file with page numbers and optimized chunking"""
        path = Path(file_path)

        if not path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        suffix = path.suffix.lower()

        if suffix == '.pdf':
            # Extract with page-by-page tracking
            page_contents, file_metadata = self._extract_pdf_with_pages(path)
        elif suffix == '.docx':
            # DOCX doesn't have reliable page numbers, use sections
            page_contents, file_metadata = self._extract_docx_with_sections(path)
        elif suffix in ['.html', '.htm']:
            text, file_metadata = self._extract_html_with_metadata(path)
            page_contents = [{'page_num': 1, 'text': text}]
        else:
            raise ValueError(f"Unsupported format: {suffix}")

        # Combine metadata
        base_metadata = {
            'source': str(path.absolute()),
            'source_type': 'file',
            'file_type': suffix[1:],
            'file_name': path.name,
            'file_size': path.stat().st_size,
            'loaded_at': datetime.now().isoformat(),
            'doc_id': self._generate_doc_id(str(path)),
            **file_metadata,
            **custom_metadata
        }

        # Create documents with page tracking
        docs_with_pages = []
        for page_info in page_contents:
            preprocessed_text = self.preprocess_text(page_info['text'])

            if preprocessed_text.strip():  # Skip empty pages
                doc = Document(
                    page_content=preprocessed_text,
                    metadata={
                        **base_metadata,
                        'page_number': page_info['page_num'],
                        'page_start': page_info['page_num'],
                        'page_end': page_info['page_num'],
                    }
                )
                docs_with_pages.append(doc)

        # Optimized chunking with page tracking
        chunks = self._optimized_split_with_pages(docs_with_pages)

        return chunks

    def _extract_pdf_with_pages(self, path: Path) -> Tuple[List[Dict], Dict]:
        """Extract PDF page-by-page"""
        doc = pymupdf.open(path)

        page_contents = []
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text()
            if page_text.strip():
                page_contents.append({
                    'page_num': page_num,
                    'text': page_text
                })

        metadata = {
            'page_count': len(doc),
            'author': doc.metadata.get('author', ''),
            'title': doc.metadata.get('title', ''),
            'subject': doc.metadata.get('subject', ''),
            'keywords': doc.metadata.get('keywords', ''),
            'creator': doc.metadata.get('creator', ''),
            'creation_date': doc.metadata.get('creationDate', ''),
            'modification_date': doc.metadata.get('modDate', ''),
        }

        doc.close()
        metadata = {k: v for k, v in metadata.items() if v}

        return page_contents, metadata

    def _extract_docx_with_sections(self, path: Path) -> Tuple[List[Dict], Dict]:
        """Extract DOCX by sections (paragraphs grouped by ~1 page worth)"""
        doc = DocxDocument(path)

        # Estimate: ~500 words per page
        WORDS_PER_PAGE = 500

        page_contents = []
        current_page_text = []
        current_word_count = 0
        page_num = 1

        for para in doc.paragraphs:
            if para.text.strip():
                para_words = len(para.text.split())
                current_page_text.append(para.text)
                current_word_count += para_words

                # When we reach ~page worth, create a new page
                if current_word_count >= WORDS_PER_PAGE:
                    page_contents.append({
                        'page_num': page_num,
                        'text': '\n\n'.join(current_page_text)
                    })
                    current_page_text = []
                    current_word_count = 0
                    page_num += 1

        # Add remaining text
        if current_page_text:
            page_contents.append({
                'page_num': page_num,
                'text': '\n\n'.join(current_page_text)
            })

        # Extract tables as separate "pages"
        for table in doc.tables:
            page_num += 1
            table_text = []
            for row in table.rows:
                row_text = ' | '.join(cell.text.strip() for cell in row.cells)
                if row_text.strip():
                    table_text.append(row_text)

            if table_text:
                page_contents.append({
                    'page_num': page_num,
                    'text': '\n'.join(table_text)
                })

        core_props = doc.core_properties
        metadata = {
            'author': core_props.author or '',
            'title': core_props.title or '',
            'subject': core_props.subject or '',
            'keywords': core_props.keywords or '',
            'created': core_props.created.isoformat() if core_props.created else '',
            'modified': core_props.modified.isoformat() if core_props.modified else '',
            'paragraph_count': len(doc.paragraphs),
            'table_count': len(doc.tables),
            'estimated_pages': len(page_contents),
        }

        metadata = {k: v for k, v in metadata.items() if v}

        return page_contents, metadata

    def _extract_html_with_metadata(self, path: Path) -> Tuple[str, Dict]:
        """Extract HTML text + metadata"""
        from bs4 import BeautifulSoup

        with open(path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')

        metadata = {
            'title': soup.title.string if soup.title else '',
            'description': '',
            'keywords': '',
            'author': '',
        }

        for meta in soup.find_all('meta'):
            name = meta.get('name', '').lower()
            content = meta.get('content', '')

            if name == 'description':
                metadata['description'] = content
            elif name == 'keywords':
                metadata['keywords'] = content
            elif name == 'author':
                metadata['author'] = content

        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()

        text = soup.get_text(separator='\n', strip=True)
        metadata = {k: v for k, v in metadata.items() if v}

        return text, metadata

    def _optimized_split(self, docs: List[Document]) -> List[Document]:
        """Split and optimize chunk sizes (for web content without pages)"""
        # Initial split
        chunks = self.splitter.split_documents(docs)

        # Optimize chunk sizes
        optimized_chunks = self._merge_small_and_split_large(chunks)

        # Add chunk metadata
        for i, chunk in enumerate(optimized_chunks):
            chunk.metadata['chunk_index'] = i
            chunk.metadata['total_chunks'] = len(optimized_chunks)
            chunk.metadata['chunk_id'] = f"{chunk.metadata.get('doc_id', 'unknown')}_chunk_{i}"
            chunk.metadata['chunk_size'] = len(chunk.page_content)

        return optimized_chunks

    def _optimized_split_with_pages(self, docs_with_pages: List[Document]) -> List[Document]:
        """Split and optimize chunk sizes while preserving page numbers"""
        # Initial split
        chunks = self.splitter.split_documents(docs_with_pages)

        # Optimize chunk sizes
        optimized_chunks = self._merge_small_and_split_large(chunks)

        # Add chunk metadata with page tracking
        for i, chunk in enumerate(optimized_chunks):
            chunk.metadata['chunk_index'] = i
            chunk.metadata['total_chunks'] = len(optimized_chunks)
            chunk.metadata['chunk_id'] = f"{chunk.metadata.get('doc_id', 'unknown')}_chunk_{i}"
            chunk.metadata['chunk_size'] = len(chunk.page_content)

            # Page numbers already preserved from original docs
            # If chunk spans multiple pages, page_start and page_end will show range

        return optimized_chunks

    def _merge_small_and_split_large(self, chunks: List[Document]) -> List[Document]:
        """Ensure all chunks are near target size"""
        optimized = []
        i = 0

        while i < len(chunks):
            current_chunk = chunks[i]
            current_size = len(current_chunk.page_content)

            # Case 1: Chunk is too small - try to merge with next
            if current_size < self.min_chunk_size and i < len(chunks) - 1:
                next_chunk = chunks[i + 1]

                # Check if we can merge without exceeding max
                combined_text = current_chunk.page_content + "\n\n" + next_chunk.page_content
                combined_size = len(combined_text)

                if combined_size <= self.max_chunk_size:
                    # Merge chunks
                    merged_metadata = current_chunk.metadata.copy()

                    # Update page range if both have page numbers
                    if 'page_number' in current_chunk.metadata and 'page_number' in next_chunk.metadata:
                        merged_metadata['page_start'] = current_chunk.metadata.get('page_start', current_chunk.metadata['page_number'])
                        merged_metadata['page_end'] = next_chunk.metadata.get('page_end', next_chunk.metadata['page_number'])
                        merged_metadata['page_number'] = f"{merged_metadata['page_start']}-{merged_metadata['page_end']}"

                    merged_chunk = Document(
                        page_content=combined_text,
                        metadata=merged_metadata
                    )
                    optimized.append(merged_chunk)
                    i += 2  # Skip both chunks
                else:
                    # Can't merge, keep as is
                    optimized.append(current_chunk)
                    i += 1

            # Case 2: Chunk is too large - split it
            elif current_size > self.max_chunk_size:
                # Re-split this chunk
                sub_chunks = self._split_large_chunk(current_chunk)
                optimized.extend(sub_chunks)
                i += 1

            # Case 3: Chunk is in acceptable range
            else:
                optimized.append(current_chunk)
                i += 1

        return optimized

    def _split_large_chunk(self, chunk: Document) -> List[Document]:
        """Split a large chunk into smaller ones"""
        # Use a temporary splitter with smaller size
        temp_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
        )

        sub_chunks = temp_splitter.split_documents([chunk])

        # Preserve original metadata
        for sub_chunk in sub_chunks:
            sub_chunk.metadata = chunk.metadata.copy()

        return sub_chunks

    def preprocess_text(self, text: str) -> str:
        """Comprehensive text preprocessing"""
        if not text:
            return ""

        if self.config['fix_encoding_errors']:
            text = self._fix_encoding_errors(text)

        if self.config['normalize_unicode']:
            text = unicodedata.normalize('NFKC', text)

        if self.config['fix_common_ocr_errors']:
            text = self._fix_ocr_errors(text)

        if self.config['remove_headers_footers']:
            text = self._remove_headers_footers(text)

        if self.config['remove_page_numbers']:
            text = self._remove_page_numbers(text)

        if self.config['remove_extra_whitespace']:
            text = re.sub(r'[ \t]+', ' ', text)

        if self.config['remove_extra_newlines']:
            text = re.sub(r'\n{3,}', '\n\n', text)

        if self.config['remove_short_lines']:
            lines = text.split('\n')
            lines = [line for line in lines if len(line.split()) >= 3 or line.strip() == '']
            text = '\n'.join(lines)

        return text.strip()

    def _fix_encoding_errors(self, text: str) -> str:
        """Fix common encoding issues"""
        replacements = {
            'â€™': "'", 'â€œ': '"', 'â€': '"',
            'â€"': '—', 'â€"': '–', 'Â': '',
            '\x00': '', '\ufffd': '',
        }
        for old, new in replacements.items():
            text = text.replace(old, new)
        return text

    def _fix_ocr_errors(self, text: str) -> str:
        """Fix common OCR errors"""
        text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
        return text

    def _remove_headers_footers(self, text: str) -> str:
        """Remove common header/footer patterns"""
        lines = text.split('\n')
        cleaned = []

        for line in lines:
            line_lower = line.lower().strip()
            skip_patterns = [
                r'^page \d+', r'^\d+ of \d+$', r'^chapter \d+',
                r'^confidential', r'^\d+$',
            ]
            if not any(re.match(p, line_lower) for p in skip_patterns):
                cleaned.append(line)

        return '\n'.join(cleaned)

    def _remove_page_numbers(self, text: str) -> str:
        """Remove standalone page numbers"""
        text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
        text = re.sub(r'\bPage\s+\d+\b', '', text, flags=re.IGNORECASE)
        return text


    def _generate_doc_id(self, source: str) -> str:
        """Generate unique document ID"""
        return hashlib.md5(source.encode()).hexdigest()[:16]


In [119]:
# Usage Examples

# Example 1: Basic usage with page numbers
loader = OptimizedPreprocessedLoader(
    chunk_size=1000,
    chunk_overlap=200,
    min_chunk_size=800,   # 80% of target
    max_chunk_size=1200   # 120% of target
)

chunks = loader.load_and_split(r"data\rag_chunking.pdf")


In [120]:
chunks

[Document(metadata={'source': 'c:\\Users\\hetba\\OneDrive\\Desktop\\Work\\Learn\\Projects\\AllinOneRAG\\data\\rag_chunking.pdf', 'source_type': 'file', 'file_type': 'pdf', 'file_name': 'rag_chunking.pdf', 'file_size': 681812, 'loaded_at': '2026-02-17T16:13:35.698502', 'doc_id': '91063af5574951e8', 'page_count': 25, 'page_number': 1, 'page_start': 1, 'page_end': 1, 'chunk_index': 0, 'total_chunks': 32, 'chunk_id': '91063af5574951e8_chunk_0', 'chunk_size': 955}, page_content="Retrieval-Augmented Generation (RAG) has emerged as a powerful technique that combines\ninformation retrieval with language generation to enhance the capabilities of large language\nmodels. At the heart of any successful RAG system lies a critical preprocessing step: chunking -\nthe process of breaking down large documents into smaller, manageable, and semantically\nChunking is not merely about dividing text; it's about preserving context, maintaining semantic\ncoherence, and optimizing information retrieval. The qu

# Embedding 

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

c:\Users\hetba\OneDrive\Desktop\Work\Learn\Projects\AllinOneRAG\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Vectore store [ Hybrid Search ]

In [6]:
api_key = os.environ["PINECONE_API_KEY"]


In [121]:
from pinecone import Pinecone, ServerlessSpec

index_name = "langchain-pinecone-hybrid-search"

pc = Pinecone(api_key=api_key)

if index_name not in pc.list_indexes().names() :
    pc.create_index(
        name = index_name,
        dimension = 384,
        metric = "dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),

    )
else :
    print(f"Index '{index_name}' already exists.")

Index 'langchain-pinecone-hybrid-search' already exists.


In [122]:
index = pc.Index(index_name)

In [9]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()


In [123]:
texts = [doc.page_content for doc in chunks]
metadatas = [doc.metadata for doc in chunks]


In [124]:
bm25_encoder.fit(texts)


  0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 32/32 [00:00<00:00, 340.69it/s]


In [125]:
vectors = []

for text, metadata in zip(texts, metadatas):
    dense_vector = embedding.embed_query(text)
    sparse_vector = bm25_encoder.encode_documents([text])[0]

    vectors.append({
        "id": metadata["chunk_id"],
        "values": dense_vector,
        "sparse_values": sparse_vector,
        "metadata": {
            **metadata,
            "text": text
        }
    })


In [126]:
BATCH_SIZE = 100
for i in range(0, len(vectors), BATCH_SIZE):
    print(f"Upserting batch {i // BATCH_SIZE + 1}")
    index.upsert(vectors=vectors[i:i + BATCH_SIZE])


Upserting batch 1


In [ ]:
# from langchain_pinecone import PineconeVectorStore

# vectorstore = PineconeVectorStore(
#     index=index,
#     embedding=embedding
# )

# vector_retriever = vectorstore.as_retriever(
#     search_kwargs={"k": 10}
# )


In [ ]:
# from langchain_community.retrievers import BM25Retriever

# bm25_retriever = BM25Retriever.from_texts(
#     [doc.page_content for doc in chunks],
#     metadatas=[doc.metadata for doc in chunks]
# )
# bm25_retriever.k = 10


In [127]:
query = "What is Chunking?"

dense_query = embedding.embed_query(query)
sparse_query = bm25_encoder.encode_queries([query])[0]


results = index.query(
    vector=dense_query,
    sparse_vector=sparse_query,
    top_k=5,
    include_metadata=True
)


In [128]:
results

QueryResponse(matches=[{'id': '91063af5574951e8_chunk_10',
 'metadata': {'chunk_id': '91063af5574951e8_chunk_10',
              'chunk_index': 10,
              'chunk_size': 993,
              'doc_id': '91063af5574951e8',
              'file_name': 'rag_chunking.pdf',
              'file_size': 681812,
              'file_type': 'pdf',
              'loaded_at': '2026-02-17T16:13:35.698502',
              'page_count': 25,
              'page_end': 7,
              'page_number': 7,
              'page_start': 7,
              'source': 'c:\\Users\\hetba\\OneDrive\\Desktop\\Work\\Learn\\Projects\\AllinOneRAG\\data\\rag_chunking.pdf',
              'source_type': 'file',
              'text': 'Hierarchical chunking creates multiple levels of chunks '
                      '- parent chunks containing broader\n'
                      'context and child chunks with specific details.\n'
                      'Documents are split into large parent chunks, which are '
                      

In [130]:
for matches in results["matches"] :
    print(matches.metadata["text"][:200])
    print(matches.metadata["source"])
    print(matches.metadata["chunk_id"])
    print(matches["score"])
    print("="*20)


Hierarchical chunking creates multiple levels of chunks - parent chunks containing broader
context and child chunks with specific details.
Documents are split into large parent chunks, which are then 
c:\Users\hetba\OneDrive\Desktop\Work\Learn\Projects\AllinOneRAG\data\rag_chunking.pdf
91063af5574951e8_chunk_10
1.64445364
TOP RAG CHUNKING METHODS YOU SHOULD
Introduction to RAG Chunking
Why Chunking Matters in RAG Systems
1. Token Limitations
2. Retrieval Precision
3. Computational Efficiency
4. Context Preservation

Fi
c:\Users\hetba\OneDrive\Desktop\Work\Learn\Projects\AllinOneRAG\data\rag_chunking.pdf
91063af5574951e8_chunk_2
1.64324689
Small-to-big chunking creates small chunks for precise retrieval but retrieves larger parent
chunks to provide comprehensive context.
Documents are split into small child chunks for indexing, but the 
c:\Users\hetba\OneDrive\Desktop\Work\Learn\Projects\AllinOneRAG\data\rag_chunking.pdf
91063af5574951e8_chunk_24
1.64208233
Instead of creating adjacent 

In [138]:
def extract_file_name(file_path):
    return os.path.basename(file_path)

file_name = extract_file_name(file_path)

# Generation

In [161]:

def get_context(question):

    dense_query = embedding.embed_query(question)
    sparse_query = bm25_encoder.encode_queries([question])[0]


    results = index.query(
        vector=dense_query,
        sparse_vector=sparse_query,
        top_k=20,
        include_metadata=True,
        filter={
            "file_name": file_name
        }
    )

    context = "\n\n".join([f"{matches.metadata['text']}\n\nMetadata: {matches.metadata}" for matches in results["matches"]])

    return  results["matches"] , context

In [132]:
from langchain_groq import ChatGroq


llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

In [133]:
# context = "\n\n".join([f"{r[0].page_content}\n\nMetadata: {r[0].metadata}" for r in results])

In [134]:



def get_llm_response(question, context):
   response = llm.invoke(f"""You are a document-grounded AI assistant.

   Answer the user's question using ONLY the information provided in <context>.

   Strict rules:
   1. Do NOT use outside knowledge.
   2. Do NOT hallucinate.
   3. If the answer is not present, respond:
      "I don't know based on the provided documents."
   4. Every factual statement MUST have a citation.
   5. Citations must reference chunk metadata (source + chunk_id).
   6. If multiple chunks support a fact, include multiple citations.
   7. Keep answers concise and factual.

   Return output in this format:

   Answer:
   <your answer here>

   Citations:
   - source: <source>, chunk_id: <chunk_id>
   - source: <source>, chunk_id: <chunk_id>

   <context>
   {context}
   </context>

   User Question:
   {question}
   """)

   return response

In [162]:
question = "Explain Chunking indetail?"

In [172]:
def rewrite_query(user_query):

    response = llm.invoke(f"""
        Rewrite this question to be optimized for semantic search in a RAG system.

        Original question:
        {user_query}

        Return ONLY the rewritten query.
    """)


    return response.content.strip()

In [173]:
rewitten_query = rewrite_query(question)

In [174]:
rewitten_query

'What is Chunking, and provide a detailed explanation of its concept, process, and applications.'

In [179]:
docs , context = get_context(rewitten_query)
response = get_llm_response(rewitten_query, context)

In [180]:
print(response.content)

Answer: 
Chunking is the process of breaking down large documents into smaller, manageable, and semantically meaningful pieces called chunks. It is a critical preprocessing step in Retrieval-Augmented Generation (RAG) systems, which combines information retrieval with language generation to enhance the capabilities of large language models.

**Concept of Chunking:**
Chunking is essential for handling large documents, as it allows for more precise retrieval, reduces computational overhead, and improves response times. The quality of the chunking strategy directly impacts the performance of the RAG system, affecting retrieval accuracy, response relevance, and computational efficiency.

**Process of Chunking:**
The chunking process involves dividing text into uniform segments based on a predetermined character or token count. There are various chunking methods, including:

1. **Fixed-Size Chunking:** divides text into uniform segments based on a predetermined character or token count.
2. 

# Reranking

In [33]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [166]:
from langchain_core.documents import Document

def pinecone_matches_to_documents(matches):
    docs = []

    for m in matches:
        docs.append(
            Document(
                page_content=m["metadata"]["text"],
                metadata=m["metadata"]
            )
        )

    return docs

rerank_context = pinecone_matches_to_documents(docs)

In [ ]:

def rerank(rerank_context) :
    pairs = [(rewitten_query, d.page_content) for d in rerank_context]

    scores = reranker.predict(pairs)

    reranked = sorted(
        zip(rerank_context, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return reranked


In [183]:
top_docs = rerank(rerank_context)[:5]

top_docs

[(Document(metadata={'chunk_id': '91063af5574951e8_chunk_0', 'chunk_index': 0, 'chunk_size': 955, 'doc_id': '91063af5574951e8', 'file_name': 'rag_chunking.pdf', 'file_size': 681812, 'file_type': 'pdf', 'loaded_at': '2026-02-17T16:13:35.698502', 'page_count': 25, 'page_end': 1, 'page_number': 1, 'page_start': 1, 'source': 'c:\\Users\\hetba\\OneDrive\\Desktop\\Work\\Learn\\Projects\\AllinOneRAG\\data\\rag_chunking.pdf', 'source_type': 'file', 'text': "Retrieval-Augmented Generation (RAG) has emerged as a powerful technique that combines\ninformation retrieval with language generation to enhance the capabilities of large language\nmodels. At the heart of any successful RAG system lies a critical preprocessing step: chunking -\nthe process of breaking down large documents into smaller, manageable, and semantically\nChunking is not merely about dividing text; it's about preserving context, maintaining semantic\ncoherence, and optimizing information retrieval. The quality of your chunking st

In [184]:
def format_reranked_docs(top_docs):
    formatted = []
    for doc, score in top_docs:
        formatted.append(f"Score: {score:.4f}\n{doc.page_content}\nMetadata: {doc.metadata}\n{'-'*50}")
    return "\n\n".join(formatted)

In [ ]:
response = get_llm_response(question, format_reranked_docs(top_docs))

In [171]:
print(response.content)

Chunking is a critical preprocessing step in Retrieval-Augmented Generation (RAG) systems that involves breaking down large documents into smaller, manageable, and semantically coherent chunks.

There are several reasons why chunking is important in RAG systems:
1. **Token Limitations**: Large language models have context window limitations and cannot process entire documents in a single pass, making chunking essential for handling large documents [91063af5574951e8_chunk_1].
2. **Retrieval Precision**: Smaller, focused chunks enable more precise retrieval, allowing RAG systems to fetch exactly the information needed to answer specific queries [91063af5574951e8_chunk_1].
3. **Computational Efficiency**: Processing smaller chunks reduces computational overhead and improves response times, making RAG systems more scalable and cost-effective [91063af5574951e8_chunk_1].
4. **Context Preservation**: Well-designed chunks maintain the semantic relationships between ideas, ensuring that retriev

# Chain

In [185]:
question = "what is Rag and chunking?"

rewitten_query = rewrite_query(question)
docs , context = get_context(rewitten_query)
response = get_llm_response(rewitten_query, context)
rerank_context = pinecone_matches_to_documents(docs)
top_docs = rerank(rerank_context)
formated_contex = format_reranked_docs(top_docs)
final_response = get_llm_response(question, formated_contex)

print(final_response.content)


RAG stands for Retrieval-Augmented Generation, which is a technique that combines information retrieval with language generation to enhance the capabilities of large language models.

Chunking is a critical preprocessing step in RAG systems, which involves breaking down large documents into smaller, manageable, and semantically meaningful pieces called chunks.

## Key Points about Chunking:

1. **Purpose**: Chunking helps in preserving context, maintaining semantic coherence, and optimizing information retrieval.
2. **Importance**: The quality of the chunking strategy directly impacts the performance of the RAG system, affecting retrieval accuracy, response relevance, and computational efficiency.
3. **Methods**: Various chunking methods exist, including:
   - **Fixed-size chunking**: divides text into uniform segments based on a predetermined character or token count.
   - **Recursive character text splitting**: a method that splits text into chunks based on characters.
   - **Semanti

# Evaluation

In [65]:
import json

with open('ragas_evaluation_qa.json', 'r') as f:
    data = json.load(f)

In [69]:
"""
Ragas Evaluation with Rate Limiting for Groq API
Fixes deprecation warnings and handles rate limits
"""

import time
import json
import os
from datasets import Dataset

# Updated imports to fix deprecation warnings
from ragas.metrics import (
    faithfulness,
    # answer_relevancy,
    context_recall,
    context_precision,
    # answer_correctness,
)
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper



# ============================================================================
# CONFIGURATION
# ============================================================================

# Rate limiting configuration
REQUESTS_PER_MINUTE = 10  # Groq free tier limit
DELAY_BETWEEN_REQUESTS = 60 / REQUESTS_PER_MINUTE  # ~2 seconds
DELAY_BETWEEN_BATCHES = 5  # Extra delay every N questions

# Batch processing configuration
BATCH_SIZE = 3  # Process 5 questions, then pause


# ============================================================================
# INITIALIZE GROQ LLM (Updated method - no deprecation)
# ============================================================================

ragas_llm = LangchainLLMWrapper(llm)


# ============================================================================
# LOAD DATA
# ============================================================================

with open('ragas_evaluation_qa.json', 'r') as f:
    data = json.load(f)

evaluation_data = {
    'user_input': [],
    'response': [],
    'retrieved_contexts': [],
    'reference': []
}


# ============================================================================
# PROCESS QUESTIONS WITH RATE LIMITING
# ============================================================================

print("Processing questions with rate limiting...")
print(f"Delay between requests: {DELAY_BETWEEN_REQUESTS:.2f}s")
print(f"Batch size: {BATCH_SIZE} questions")
print(f"Extra delay between batches: {DELAY_BETWEEN_BATCHES}s\n")

total_questions = len(data['evaluation_dataset'])

for idx, item in enumerate(data['evaluation_dataset'], 1):
    question = item['question']
    ground_truth = item['ground_truth']

    print(f"[{idx}/{total_questions}] Processing: {question[:60]}...")

    # YOUR RAG PIPELINE WITH RETRY LOGIC
    max_retries = 3
    retry_count = 0

    while retry_count < max_retries:
        try:
            # Add delay before each API call
            if idx > 1:  # Don't delay the first request
                time.sleep(DELAY_BETWEEN_REQUESTS)

            # Get context and response from your RAG pipeline
            context = get_context(question)
            response = get_llm_response(question, context)

            print("="*50)
            print("Question:", question)
            print("Context:", context)
            print("Answer:", response.content)
            print("="*50)


            # Success - break the retry loop
            break

        except Exception as e:
            retry_count += 1
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = 60 * retry_count  # Exponential backoff
                print(f"  ⚠ Rate limit hit. Waiting {wait_time}s before retry {retry_count}/{max_retries}...")
                time.sleep(wait_time)
            else:
                print(f"  ⚠ Error: {str(e)}")
                if retry_count >= max_retries:
                    print(f"  ❌ Failed after {max_retries} retries. Skipping question.")
                    # Use placeholder data if all retries fail
                    context = ["Error: Could not retrieve context"]
                    response = type('obj', (object,), {'content': 'Error: Could not generate response'})()
                    break
                time.sleep(5)

    # CRITICAL FIX: Ensure context is a LIST of strings
    if isinstance(context, str):
        context = [context]
    elif not isinstance(context, list):
        context = [str(context)]

    context = [str(ctx) if not isinstance(ctx, str) else ctx for ctx in context]

    # Append to evaluation data
    evaluation_data['user_input'].append(question)
    evaluation_data['response'].append(str(response.content))
    evaluation_data['retrieved_contexts'].append(context)
    evaluation_data['reference'].append(ground_truth)

    # Add extra delay after every batch
    if idx % BATCH_SIZE == 0 and idx < total_questions:
        print(f"  💤 Batch complete. Resting for {DELAY_BETWEEN_BATCHES}s...\n")
        time.sleep(DELAY_BETWEEN_BATCHES)

print(f"\n✓ Processed all {total_questions} questions successfully!\n")


# ============================================================================
# CREATE DATASET AND EVALUATE
# ============================================================================

dataset = Dataset.from_dict(evaluation_data)

print("Sample data format:")
print(f"Question: {dataset[0]['user_input'][:60]}...")
print(f"Answer: {dataset[0]['response'][:60]}...")
print(f"Contexts type: {type(dataset[0]['retrieved_contexts'])}")
print(f"Number of contexts: {len(dataset[0]['retrieved_contexts'])}")
print()

print("Starting Ragas evaluation...")
print("Note: This will also make API calls and may take time due to rate limits.\n")

# Evaluate with rate-limited execution
results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        # answer_relevancy,
        context_recall,
        context_precision,
        # answer_correctness
    ],
    llm=ragas_llm,
    raise_exceptions=False,  # Continue even if some evaluations fail
)


C:\Users\hetba\AppData\Local\Temp\ipykernel_5268\4273459163.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\hetba\AppData\Local\Temp\ipykernel_5268\4273459163.py:12: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\hetba\AppData\Local\Temp\ipykernel_5268\4273459163.py:12: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\hetba\AppData\Local\Temp\ipykernel_5268\4

Processing questions with rate limiting...
Delay between requests: 6.00s
Batch size: 3 questions
Extra delay between batches: 5s

[1/10] Processing: What are the three core pipelines in the FTI (Feature/Traini...
Question: What are the three core pipelines in the FTI (Feature/Training/Inference) architecture and what does each one do?
Context: we want to focus on the core functionality. When implementing each component, we will look 
into all the little details. But for now, the fundamental question we must ask ourselves is this: 
How can we apply the FTI pipeline design to implement the preceding list of requirements?
How to design the LLM Twin architecture using the FTI 
We will split the system into four core components. You will ask yourself this: “Four? Why not 
three, as the FTI pipeline design clearly states?” That is a great question. Fortunately, the answer 
is simple. We must also implement the data pipeline along the three feature/training/inference 
pipelines. According to 

Evaluating:  90%|█████████ | 27/30 [03:01<00:27,  9.19s/it]Exception raised in Job[13]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Evaluating: 100%|██████████| 30/30 [03:19<00:00,  6.65s/it]


In [70]:
# ============================================================================
# DISPLAY RESULTS
# ============================================================================

print("\n" + "="*70)
print(" "*25 + "RAGAS EVALUATION RESULTS")
print("="*70)

# Handle NaN values in results
def safe_score(score):
    """Return score if valid, otherwise return 'N/A'"""
    try:
        if isinstance(score, (int, float)) and not (score != score):  # Check for NaN
            return f"{score}"
        return "N/A"
    except:
        return "N/A"

print(f"\nOverall Scores:")
print(f"  Faithfulness:        {(results['faithfulness'])}")
print(f"  Context Precision:   {(results['context_precision'])}")
print(f"  Context Recall:      {(results['context_recall'])}")
print("\n" + "="*70)

# Convert to DataFrame and display
results_df = results.to_pandas()

# Filter out NaN columns for display
display_cols = ['user_input', 'faithfulness',
                'context_precision', 'context_recall']
available_cols = [col for col in display_cols if col in results_df.columns]

print("\nDetailed Results by Question:")
print(results_df[available_cols].to_string(max_colwidth=60))

# Save results
output_file = 'ragas_evaluation_results.csv'
results_df.to_csv(output_file, index=False)
print(f"\n✓ Results saved to: {output_file}")

# Calculate statistics on valid scores only
print("\nStatistics (excluding NaN/failed evaluations):")
for metric in ['faithfulness', 'context_precision',
               'context_recall']:
    if metric in results_df.columns:
        valid_scores = results_df[metric].dropna()
        if len(valid_scores) > 0:
            print(f"  {metric:20s}: mean={valid_scores.mean():.4f}, "
                  f"min={valid_scores.min():.4f}, max={valid_scores.max():.4f}, "
                  f"valid={len(valid_scores)}/{len(results_df)}")
        else:
            print(f"  {metric:20s}: No valid scores")

print("\n" + "="*70)


                         RAGAS EVALUATION RESULTS

Overall Scores:
  Faithfulness:        [1.0, 1.0, 1.0, 1.0, 0.8, 0.7, 1.0, 1.0, 1.0, 1.0]
  Context Precision:   [0.9999999999, nan, 0.9999999999, 0.9999999999, 0.9999999999, 0.9999999999, nan, 0.9999999999, 0.9999999999, 0.0]
  Context Recall:      [1.0, 1.0, 1.0, 1.0, nan, 1.0, 1.0, 1.0, 1.0, 1.0]


Detailed Results by Question:
                                                    user_input  faithfulness  context_precision  context_recall
0  What are the three core pipelines in the FTI (Feature/Tr...           1.0                1.0             1.0
1  What is an LLM Twin and what is its primary use case in ...           1.0                NaN             1.0
2  According to the book, what are the main benefits of usi...           1.0                1.0             1.0
3  What are the four main components of the LLM Twin system...           1.0                1.0             1.0
4  What is ZenML and what are its three main features u